In [ ]:
# use conda to create an environment with openmm and some dependencies
# conda create -n openmm-test -c conda-forge python openmm jupyterlab ipykernel parmed pdb2pqr
# which python
# then i had to install the WSL extension in VS code, run VS Code from WSL using code ., and trust the folder
# test the openmm installation 

import openmm as mm
from openmm import testInstallation
testInstallation.main()

# check GPU and install new drivers if needed
!nvidia-smi


OpenMM Version: 8.6.1
Git Revision: b399af4725573963b46d6c1083fdcf7a37615857

There are 3 Platforms available:

1 Reference - Successfully computed forces
2 CPU - Successfully computed forces
3 CUDA - Successfully computed forces

Median difference in forces between platforms:

Reference vs. CPU: 6.28735e-06
Reference vs. CUDA: 6.72624e-06
CPU vs. CUDA: 7.68204e-07

All differences are within tolerance.
Tue Nov  3 17:12:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 620.67.01              KMD Version: 621.01        CUDA UMD Version: 13.5     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=======

In [3]:
# purpose: set up an md simulation with a dimer apoenzyme (no ligand) at pH 7.4
# first, protonate the protein using pdb2pqr and propka

!pdb2pqr --ff=AMBER --titration-state-method=propka --with-ph=7.4 pvivax-6pgdh-dimer-relaxed.pdb --ffout=AMBER pvivax-6pgdh-dimer-relaxed_pH7-4.pdb

INFO:PDB2PQR v3.8.0: biomolecular structure conversion software.
INFO:Please cite:  Jurrus E, et al.  Improvements to the APBS biomolecular solvation software suite.  Protein Sci 27 112-128 (2018).
INFO:Please cite:  Dolinsky TJ, et al.  PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. Nucleic Acids Res 35 W522-W525 (2007).
INFO:Checking and transforming input arguments.
INFO:Loading topology files.
INFO:Loading molecule: pvivax-6pgdh-dimer-relaxed.pdb
INFO:Setting up molecule.
INFO:Created biomolecule object with 946 residues and 15050 atoms.
INFO:Setting termini states for biomolecule chains.
INFO:Loading forcefield.
INFO:Loading hydrogen topology definitions.
INFO:This biomolecule is clean.  No repair needed.
INFO:Updating disulfide bridges.
INFO:Debumping biomolecule.
INFO:Assigning titration states with PROPKA.
INFO:
propka3.5.1                                                                                  2026-11-03

-

In [ ]:
# import other packages and 
import openmm as mm
import openmm.app as app
import openmm.unit as unit
import parmed as pmd

# create the topology and system using the protonated PDB file
print('Building OpenMM system...')
enzyme_pdb = app.PDBFile('pvivax-6pgdh-dimer-relaxed_pH7-4.pdb')
modeller = app.Modeller(enzyme_pdb.topology, enzyme_pdb.positions)
forcefield = app.ForceField('amber19-all.xml', 'amber19/opc3.xml')

# add any needed hydrogens
# addhydrogens will not overwrite existing hydrogens, so it is safe to call it even if the PDB already has hydrogens
def n_h(top):
    return sum(1 for a in top.atoms() if a.element is not None and a.element.symbol == 'H')
print(n_h(modeller.topology))
modeller.addHydrogens(forcefield, pH=7.4)
print(n_h(modeller.topology))

Building OpenMM system...
7562
7562


In [8]:
# validate the protonation state of histidine and glutamate residues

print("--- His/Glu states of enzyme after addHydrogens ---")
for res in modeller.topology.residues():
    if res.name in ('HIS', 'HID', 'HIE', 'HIP', 'GLU', 'GLH'):
        atom_names = [a.name for a in res.atoms()]
        print(res.name, res.id, atom_names)

--- His/Glu states of enzyme after addHydrogens ---
GLU 38 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 41 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 49 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 58 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 61 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 85 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
HIS 93 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'H', 'HA', 'HB2', 'HB3', 'CD2', 'ND1', 'CE1', 'NE2', 'HD1', 'HD2', 'HE1']
GLU 95 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 96 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 106 ['N', 'CA', 'C', 'CB', 'O

In [9]:
# validate disulfide bonds if they are important for your protein
for bond in modeller.topology.bonds():
    atom1, atom2 = bond
    if atom1.name == 'SG' and atom2.name == 'SG':
        print(f"  Bonded: {atom1.residue.name}{atom1.residue.id} to {atom2.residue.name}{atom2.residue.id}")

n_bonds_before = __builtins__.sum(1 for _ in modeller.topology.bonds())
modeller.topology.createDisulfideBonds(modeller.positions)
n_bonds_after = __builtins__.sum(1 for _ in modeller.topology.bonds())
print(f"Disulfide bonds added: {n_bonds_after - n_bonds_before}")

Disulfide bonds added: 0


In [13]:
# create an octahedral box, add solvent (0.15 M NaCl), and neutralize charges.
# note that model='tip3p' because model in openMM does not take opc as an argument. however, this just tells openMM that you're using a three point water system, and the actual parameters for opc water are set in the forcefield above.
modeller.addSolvent(forcefield, model='tip3p', boxShape='octahedron', padding=1.0*unit.nanometers, positiveIon='Na+', negativeIon='Cl-', ionicStrength=0.15*unit.molar, neutralize=True)

# 4. create topology, add temperature control then barostat, save solvated system. also, create a separate system export to eventually save as a .prmtop file. for some reason, hydrogen bond occupancy didn't work with the solvated_system.pdb file, although other analysis did. constraints and rigidwater have to be removed for that due to incompatibilities between openmm and parmed.
system = forcefield.createSystem(
    modeller.topology, 
    nonbondedMethod=app.PME, 
    nonbondedCutoff=1.0*unit.nanometer, 
    constraints=app.HBonds,
    rigidWater=True
)

system_for_export = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints=None,
    rigidWater=False
)

# export the system. note that cif is a substitute for pdb when the atom count is >99,999.
# also exporting a prmtop file without constraints for some downstream analysis, such as hydrogen bond occupancy.
from openmm.app import PDBxFile
PDBxFile.writeFile(modeller.topology, modeller.positions, open('solvated_system.cif', 'w'))
print(".cif file exported for downstream analysis.")
structure = pmd.openmm.load_topology(modeller.topology, system_for_export, modeller.positions)
structure.save('solvated_system.prmtop', format='amber')
structure.save('solvated_system.inpcrd', format='rst7')
print(".prmtop file exported for downstream analysis.")

.cif file exported for downstream analysis.
.prmtop file exported for downstream analysis.


In [ ]:
# 5. restrain heavy atoms by applying a penalty to their harmonic motion, because this allows for the water molecules to settle into the system and can help prevent sudden jumps in energy or unexpected changes in structure.
print("Step 1: Applying positional restraints before equilibration...")
restraint = mm.CustomExternalForce('k*periodicdistance(x, y, z, x0, y0, z0)^2')
restraint.addGlobalParameter('k', 100.0 * unit.kilocalories_per_mole/unit.angstrom**2)
restraint.addPerParticleParameter('x0')
restraint.addPerParticleParameter('y0')
restraint.addPerParticleParameter('z0')

restrained_atom_indices = []
for atom in modeller.topology.atoms():
    if atom.residue.name in ('HOH', 'WAT', 'NA', 'CL', 'K'):
        continue
    if atom.element is not None and atom.element.symbol != 'H':
        restraint.addParticle(atom.index, modeller.positions[atom.index])
        restrained_atom_indices.append(atom.index)

system.addForce(restraint)
print(f"Restrained {len(restrained_atom_indices)} heavy atoms.")

Step 1: Applying positional restraints before equilibration...
Restrained 7492 heavy atoms.


In [17]:
water_count = sum(1 for res in modeller.topology.residues() if res.name in ('HOH', 'WAT'))
print(f"Water molecules in the system: {water_count}")

Water molecules in the system: 217497


In [15]:
# 6. initializes the integrator, which sets initial temperature. change the random seed (set for reproducibility) if desired for doing replicates.
# CUDA ensures that the GPU will be used for computation. mixed precision is sufficient for most cases. higher precision is much more computationally expensive.
integrator = mm.LangevinMiddleIntegrator(0*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
seed=13
integrator.setRandomNumberSeed(seed)
print(f"Random seed: {seed}")
platform = mm.Platform.getPlatformByName('CUDA')
properties = {'CudaPrecision': 'mixed'}
simulation = app.Simulation(modeller.topology, system, integrator, platform, properties)
simulation.context.setPositions(modeller.positions)

# 7. initialize simulation: perform energy minimization. report energy before and after minimization. 
print("Step 2: Energy minimization and positional restraints...")
state_before = simulation.context.getState(getEnergy=True)
print(f"Potential energy before minimization: {state_before.getPotentialEnergy()}")
simulation.minimizeEnergy(maxIterations=10000)
state_after = simulation.context.getState(getEnergy=True)
print(f"Potential energy after minimization: {state_after.getPotentialEnergy()}")

Random seed: 13
Step 2: Energy minimization and positional restraints...
Potential energy before minimization: -8011058.59772715 kJ/mol
Potential energy after minimization: -13167326.45191563 kJ/mol


In [16]:
# 8. gradual heating was chosen for a similar reason as heavy atom restraints. this simulation is now NVT.
# moving the system from its crystallized structure to 300 K abruptly can cause unfolding or sudden unwanted shifts in structure, so this allows the system to settle more gently.
print("Step 3: NVT simulation: gradual heating phase (0 K -> 300 K over 400 ps)...")
heating_steps = 100
steps_per_increment = 2000  # 2000 steps * 2 fs = 4 ps per temperature step, allows the system to slowly heat up
for i in range(heating_steps):
    current_temp = (i / heating_steps) * 300 * unit.kelvin
    integrator.setTemperature(current_temp)
    simulation.step(steps_per_increment)
    if i % 10 == 0:
        print(f"  -> Temperature reached: {current_temp.value_in_unit(unit.kelvin):.1f} K")
integrator.setTemperature(300*unit.kelvin)

# 9. add barostat (which sets the pressure of the system). 
# do not add the barostat until the temperature is fully set after gradual heating to prevent large changes in volume this simulation is now NPT.
system.addForce(mm.MonteCarloBarostat(1.0*unit.bar, 300*unit.kelvin))
simulation.context.reinitialize(preserveState=True)
print("NPT simulation initialized with barostat added.")


Step 3: NVT simulation: gradual heating phase (0 K -> 300 K over 400 ps)...
  -> Temperature reached: 0.0 K
  -> Temperature reached: 30.0 K
  -> Temperature reached: 60.0 K
  -> Temperature reached: 90.0 K
  -> Temperature reached: 120.0 K
  -> Temperature reached: 150.0 K
  -> Temperature reached: 180.0 K
  -> Temperature reached: 210.0 K
  -> Temperature reached: 240.0 K
  -> Temperature reached: 270.0 K
NPT simulation initialized with barostat added.


In [ ]:
# 10. release restraints on heavy atoms gradually
# release time of 1 ns, since it's 10 steps, and 100 ps per step
print("Step 4: Releasing positional restraints gradually...")
release_steps = 10
initial_k = 100.0
for i in range(release_steps):
    scale = 1.0 - (i / (release_steps - 1))  # 1.0 -> 0.0
    simulation.context.setParameter('k', initial_k * scale * unit.kilocalories_per_mole/unit.angstrom**2)
    simulation.step(50000)  # 100 ps per stage at 2 fs/step
    print(f"  -> Restraint scale: {scale:.2f}")

print("System equilibrated at 300 K, restraints released. Starting 10 ns production run...")

# 11. set up reporters. downstream analysis such as RMSD and RMSF. will be done on the trajectory.dcd and .prmtop files. run simulation for 10 ns (5,000,000 steps * 0.002 ps). note that this is only a test run, and 100 ns+ (and/or doing replicates of 100 ns+) are strongly recommended for publishable results.
# to increase time, increase both totalSteps and simulation.step
simulation.reporters.append(app.DCDReporter('trajectory.dcd', 10000)) 
simulation.reporters.append(app.CheckpointReporter('checkpoint.chk', 10000))
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, 10000, step=True, potentialEnergy=True, 
    temperature=True, progress=True, totalSteps=5000000, speed=True
))

simulation.step(5000000)
print("Simulation complete.")